In [ ]:
import json
import os

import requests


# ── Configuration ──────────────────────────────────────────────────────────────
INPUT_DIR       = "main_jsons"          # merged-chunk JSONs produced by chunks.ipynb
OUTPUT_DIR      = "main_embedded_jsons" # embedded JSONs (vector store)
OLLAMA_URL      = "http://localhost:11434/api/embed"
EMBEDDING_MODEL = "bge-m3"              # must be pulled in Ollama: `ollama pull bge-m3`


def create_embeddings(text_list):
    """
    Send a list of texts to the Ollama embedding API and return their vectors.
    """
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": EMBEDDING_MODEL,
            "input": text_list,
        },
        timeout=300,
    )
    response.raise_for_status()
    return response.json()["embeddings"]


def process_json(json_path):
    """
    Read one merged-chunk JSON file, generate embeddings for every chunk,
    and save the updated file to OUTPUT_DIR.
    Skips files that have already been processed.
    """
    filename    = os.path.basename(json_path)
    output_path = os.path.join(OUTPUT_DIR, filename)

    # Skip if already embedded
    if os.path.exists(output_path):
        print(f"Skipping (already embedded): {filename}")
        return

    print("\n" + "=" * 70)
    print(f"Processing: {filename}")
    print("=" * 70)

    with open(json_path, "r", encoding="utf-8") as file:
        content = json.load(file)

    chunks = content["chunks"]
    print(f"Total chunks: {len(chunks)}")

    texts = [chunk["text"] for chunk in chunks]

    print("Creating embeddings...")
    embeddings = create_embeddings(texts)
    print(f"Embeddings created: {len(embeddings)}")

    if len(chunks) != len(embeddings):
        raise ValueError(
            f"Chunk count ({len(chunks)}) does not match "
            f"embedding count ({len(embeddings)})"
        )

    for i, chunk in enumerate(chunks):
        chunk["chunk_number"] = i + 1
        chunk["embedding"]    = embeddings[i]
        print(f"  Chunk {i + 1}/{len(chunks)} -> embedding added")

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(content, file, ensure_ascii=False, indent=4)

    print(f"Saved: {output_path}  (total embeddings: {len(embeddings)})")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    json_files = sorted(
        filename
        for filename in os.listdir(INPUT_DIR)
        if filename.lower().endswith(".json")
    )

    print(f"Found {len(json_files)} JSON files in '{INPUT_DIR}'.")

    for index, filename in enumerate(json_files, start=1):
        print(f"\nFile {index}/{len(json_files)}")
        json_path = os.path.join(INPUT_DIR, filename)
        try:
            process_json(json_path)
        except Exception as error:
            print(f"\nError processing: {filename}")
            print(f"Error: {error}")

    print("\n" + "=" * 70)
    print("All JSON files processed.")
    print("=" * 70)


if __name__ == "__main__":
    main()
